# Chess Piece Detection — Full Evaluation
**Custom CNN vs YOLOv8** on `koryakinp/chess-positions` test split

Models: [`honi05/chess-piece-cnn`](https://huggingface.co/honi05/chess-piece-cnn) · [`honi05/chess-piece-yolo`](https://huggingface.co/honi05/chess-piece-yolo)  
Dataset: [`honi05/chess-positions-cv`](https://huggingface.co/datasets/honi05/chess-positions-cv) (derived from Kaggle `koryakinp/chess-positions`)  

**Runtime required:** GPU · T4 (Runtime → Change runtime type → T4 GPU)

---
### Task overview
| | CNN | YOLO |
|---|---|---|
| Input | 50×50 cell crop | 400×400 board image |
| Output | 13-class label (empty / 12 piece types) | Bounding boxes with 12-class piece labels |
| Test set | 64 cells × N boards | N full board images |

**Metrics computed:** Accuracy · Precision · Recall · F1 (macro/weighted/per-class) · Confusion matrix · mAP@50 · mAP@50-95 · Latency

## 1 · GPU verification

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime → Change runtime type → T4 GPU and re-run."
)

props = torch.cuda.get_device_properties(0)
print(f"GPU  : {props.name}")
print(f"VRAM : {props.total_memory / 1e9:.1f} GB")
print(f"CUDA : {torch.version.cuda}")
print(f"PyTorch : {torch.__version__}")

DEVICE = torch.device("cuda")

## 2 · Install dependencies

In [ ]:
%pip install -q \
    ultralytics \
    huggingface_hub \
    kagglehub \
    thop \
    scikit-learn \
    seaborn \
    matplotlib \
    Pillow \
    tqdm \
    pyyaml

print("Dependencies installed.")

## 3 · Kaggle credentials

**Option A (recommended):** Add your Kaggle username and key as Colab Secrets  
(*Secrets* panel on the left sidebar, key icon → add `KAGGLE_USERNAME` and `KAGGLE_KEY`).

**Option B:** Paste them directly in the cell below (less secure).

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"]      = userdata.get("KAGGLE_KEY")
    print("Kaggle credentials loaded from Colab Secrets.")
except Exception:
    # Fallback: paste directly
    os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USERNAME"   # <-- edit
    os.environ["KAGGLE_KEY"]      = "YOUR_KAGGLE_API_KEY"    # <-- edit
    print("Using hard-coded credentials (update before running).")

## 4 · Download dataset (Kaggle) + trained models (HuggingFace)

In [ ]:
import kagglehub
from pathlib import Path

print("Downloading koryakinp/chess-positions from Kaggle...")
raw_root = Path(kagglehub.dataset_download("koryakinp/chess-positions"))
print(f"  Cached at: {raw_root}")

def _find_split(root: Path, name: str) -> Path | None:
    direct = root / name
    if direct.is_dir():
        return direct
    hits = list(root.rglob(name))
    return hits[0] if hits else None

TEST_DIR  = _find_split(raw_root, "test")
TRAIN_DIR = _find_split(raw_root, "train")

assert TEST_DIR  is not None, f"Could not find test/  under {raw_root}"
assert TRAIN_DIR is not None, f"Could not find train/ under {raw_root}"

def _count_imgs(d: Path) -> int:
    return sum(1 for ext in ("*.png", "*.jpeg", "*.jpg") for _ in d.glob(ext))

n_train = _count_imgs(TRAIN_DIR)
n_test  = _count_imgs(TEST_DIR)
print(f"  Train: {TRAIN_DIR.name}  ({n_train:,} images)")
print(f"  Test : {TEST_DIR.name}   ({n_test:,} images)")

TEST_IMGS = sorted(
    list(TEST_DIR.glob("*.png")) +
    list(TEST_DIR.glob("*.jpeg")) +
    list(TEST_DIR.glob("*.jpg"))
)
print(f"\nTest images ready: {len(TEST_IMGS):,}")

In [ ]:
from huggingface_hub import hf_hub_download

print("Downloading CNN model from HuggingFace...")
CNN_WEIGHTS = Path(hf_hub_download("honi05/chess-piece-cnn", "chess_cnn.pt"))
CNN_CONFIG  = Path(hf_hub_download("honi05/chess-piece-cnn", "chess_cnn_config.json"))
print(f"  Weights : {CNN_WEIGHTS}")
print(f"  Config  : {CNN_CONFIG}")

print("\nDownloading YOLO models from HuggingFace...")
YOLO_N_WEIGHTS    = Path(hf_hub_download("honi05/chess-piece-yolo", "yolov8n/best.pt"))
YOLO_S_WEIGHTS    = Path(hf_hub_download("honi05/chess-piece-yolo", "yolov8s/best.pt"))
YOLO_PICO_WEIGHTS = Path(hf_hub_download("honi05/chess-piece-yolo", "yolo_pico/best.pt"))
print(f"  yolov8n  : {YOLO_N_WEIGHTS}")
print(f"  yolov8s  : {YOLO_S_WEIGHTS}")
print(f"  yolo_pico: {YOLO_PICO_WEIGHTS}")

## 5 · Project code (inlined — no local package required)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from PIL import Image

# ── Constants ────────────────────────────────────────────────────────────────
CNN_CLASSES = [
    "empty",
    "wP", "wN", "wB", "wR", "wQ", "wK",
    "bP", "bN", "bB", "bR", "bQ", "bK",
]
YOLO_CLASSES = CNN_CLASSES[1:]   # 12 classes — no empty
FEN_TO_CNN = {
    "P": 1, "N": 2, "B": 3, "R": 4, "Q": 5, "K": 6,
    "p": 7, "n": 8, "b": 9, "r": 10, "q": 11, "k": 12,
}
CNN_TO_YOLO = {i: i - 1 for i in range(1, 13)}  # CNN idx → YOLO idx

IMG_SIZE = 400
GRID     = 8
CELL     = IMG_SIZE // GRID   # 50 px


# ── Model ────────────────────────────────────────────────────────────────────
class ChessCNN(nn.Module):
    """Configurable CNN for 50×50 RGB cell crops (from-scratch)."""
    def __init__(self, num_classes=13, channels=(32, 64, 128),
                 fc_dim=256, dropout=0.3):
        super().__init__()
        layers, in_c = [], 3
        for out_c in channels:
            layers += [
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            ]
            in_c = out_c
        self.features    = nn.Sequential(*layers)
        self.pool        = nn.AdaptiveAvgPool2d(1)
        self.classifier  = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(in_c, fc_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(fc_dim, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


# ── Image / FEN helpers ───────────────────────────────────────────────────────
def load_image(path) -> np.ndarray:
    return np.asarray(Image.open(path).convert("RGB"))


def slice_cells(img: np.ndarray) -> np.ndarray:
    """400×400×3 board → (64, 50, 50, 3) row-major."""
    cells = np.empty((GRID * GRID, CELL, CELL, 3), dtype=img.dtype)
    k = 0
    for r in range(GRID):
        for c in range(GRID):
            cells[k] = img[r*CELL:(r+1)*CELL, c*CELL:(c+1)*CELL]
            k += 1
    return cells


def filename_to_fen(path) -> str:
    return Path(path).stem


def fen_to_label_grid(fen: str) -> list[list[str]]:
    grid = []
    for rank in fen.split("-"):
        row = []
        for ch in rank:
            if ch.isdigit():
                row.extend([""] * int(ch))
            else:
                row.append(ch)
        grid.append(row)
    return grid


def label_grid_to_cnn_indices(grid) -> np.ndarray:
    idx = np.zeros((8, 8), dtype=np.int64)
    for r in range(8):
        for c in range(8):
            ch = grid[r][c]
            idx[r, c] = FEN_TO_CNN[ch] if ch else 0
    return idx


def board_cnn_labels(img_path) -> np.ndarray:
    """Returns flat int64 array of 64 CNN labels for one board."""
    fen   = filename_to_fen(img_path)
    grid  = fen_to_label_grid(fen)
    idx   = label_grid_to_cnn_indices(grid)
    return idx.reshape(-1)


def board_yolo_lines(img_path) -> list[str]:
    """Returns YOLO-format label lines for one board image."""
    fen  = filename_to_fen(img_path)
    grid = fen_to_label_grid(fen)
    idx  = label_grid_to_cnn_indices(grid)
    lines = []
    for r in range(GRID):
        for c in range(GRID):
            cnn = int(idx[r, c])
            if cnn == 0:
                continue
            xc = (c * CELL + CELL / 2) / IMG_SIZE
            yc = (r * CELL + CELL / 2) / IMG_SIZE
            w  = h = CELL / IMG_SIZE
            lines.append(f"{CNN_TO_YOLO[cnn]} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
    return lines


print(f"CNN classes : {CNN_CLASSES}")
print(f"YOLO classes: {YOLO_CLASSES}")

## 6 · Build YOLO test-split directory (images + labels)

In [ ]:
import shutil, yaml
from tqdm.auto import tqdm

YOLO_DS   = Path("/content/yolo_dataset")
YOLO_IMGS = YOLO_DS / "images" / "test"
YOLO_LBLS = YOLO_DS / "labels" / "test"
YOLO_IMGS.mkdir(parents=True, exist_ok=True)
YOLO_LBLS.mkdir(parents=True, exist_ok=True)

print(f"Writing YOLO labels for {len(TEST_IMGS):,} test images...")
for img_path in tqdm(TEST_IMGS):
    # Symlink / copy image
    dst_img = YOLO_IMGS / img_path.name
    if not dst_img.exists():
        os.symlink(img_path, dst_img)   # symlink avoids duplicating 4 GB of images
    # Label file
    lbl = YOLO_LBLS / f"{img_path.stem}.txt"
    if not lbl.exists():
        lbl.write_text("\n".join(board_yolo_lines(img_path)))

# data.yaml for ultralytics val()
DATA_YAML = YOLO_DS / "data.yaml"
DATA_YAML.write_text(yaml.safe_dump({
    "path" : str(YOLO_DS),
    "train": "images/train",   # unused here
    "val"  : "images/test",
    "names": {i: n for i, n in enumerate(YOLO_CLASSES)},
}))
print(f"data.yaml  → {DATA_YAML}")
print("YOLO dataset ready.")

## 7 · CNN evaluation

### 7.1 — Load model

In [ ]:
import json

cfg = json.loads(CNN_CONFIG.read_text())
bp  = cfg["best_params"]

n_blocks = bp["n_blocks"]
base_ch  = bp["base_channels"]
channels = tuple(base_ch * (2 ** i) for i in range(n_blocks))
fc_dim   = bp["fc_dim"]
dropout  = bp["dropout"]

cnn = ChessCNN(num_classes=13, channels=channels, fc_dim=fc_dim, dropout=dropout)
cnn.load_state_dict(torch.load(CNN_WEIGHTS, map_location=DEVICE))
cnn = cnn.to(DEVICE).eval()

n_params = sum(p.numel() for p in cnn.parameters())
print(f"Loaded ChessCNN")
print(f"  Architecture : channels={channels}, fc_dim={fc_dim}, dropout={dropout}")
print(f"  Parameters   : {n_params:,}")

### 7.2 — Run inference on test cells

In [ ]:
import time

@torch.no_grad()
def run_cnn_inference(
    model,
    img_paths,
    device,
    batch_size: int = 1024,
    max_empty_ratio: float | None = 1.0,
):
    """
    Iterate over boards, slice cells, run CNN in batches.
    Returns (all_preds, all_labels) as numpy int arrays.
    max_empty_ratio: cap empty cells to this × occupied count per board.
    """
    model.eval()
    all_preds, all_labels = [], []
    buf_cells, buf_labels = [], []

    def flush():
        if not buf_cells:
            return
        x = torch.stack(buf_cells).to(device)
        p = model(x).argmax(1).cpu().numpy()
        all_preds.extend(p.tolist())
        all_labels.extend(buf_labels)
        buf_cells.clear()
        buf_labels.clear()

    for img_path in tqdm(img_paths, desc="CNN inference"):
        img   = load_image(img_path)
        cells = slice_cells(img)          # (64, 50, 50, 3)
        lbls  = board_cnn_labels(img_path)  # (64,)

        piece_idx = [i for i, l in enumerate(lbls) if l != 0]
        empty_idx = [i for i, l in enumerate(lbls) if l == 0]

        if max_empty_ratio is not None:
            cap = max(1, int(len(piece_idx) * max_empty_ratio))
            empty_idx = empty_idx[:cap]

        for ci in piece_idx + empty_idx:
            cell = torch.from_numpy(
                cells[ci].astype(np.float32) / 255.0
            ).permute(2, 0, 1)
            buf_cells.append(cell)
            buf_labels.append(int(lbls[ci]))
            if len(buf_cells) >= batch_size:
                flush()

    flush()
    return np.array(all_preds, dtype=np.int64), np.array(all_labels, dtype=np.int64)


print("Running CNN on full test set (this takes ~2-4 min on T4)...")
t0 = time.perf_counter()
cnn_preds, cnn_labels = run_cnn_inference(cnn, TEST_IMGS, DEVICE)
cnn_eval_time = time.perf_counter() - t0

print(f"Done in {cnn_eval_time:.1f}s")
print(f"Total cell samples : {len(cnn_preds):,}")
print(f"  Occupied (label≠0): {(cnn_labels != 0).sum():,}")
print(f"  Empty   (label=0) : {(cnn_labels == 0).sum():,}")

### 7.3 — Classification metrics

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

# ── Full 13-class metrics ─────────────────────────────────────────────────────
acc_full   = accuracy_score(cnn_labels, cnn_preds)
f1_macro   = f1_score(cnn_labels, cnn_preds, average="macro",    labels=list(range(13)), zero_division=0)
f1_weighted= f1_score(cnn_labels, cnn_preds, average="weighted", labels=list(range(13)), zero_division=0)
prec_macro = precision_score(cnn_labels, cnn_preds, average="macro",    labels=list(range(13)), zero_division=0)
rec_macro  = recall_score(cnn_labels, cnn_preds,   average="macro",    labels=list(range(13)), zero_division=0)

# ── Occupied-only metrics (labels 1-12) ──────────────────────────────────────
occ_mask     = cnn_labels != 0
occ_preds    = cnn_preds[occ_mask]
occ_labels   = cnn_labels[occ_mask]
acc_occ      = accuracy_score(occ_labels, occ_preds)
f1_occ_macro = f1_score(occ_labels, occ_preds, average="macro",    labels=list(range(1,13)), zero_division=0)
f1_occ_w     = f1_score(occ_labels, occ_preds, average="weighted", labels=list(range(1,13)), zero_division=0)
prec_occ     = precision_score(occ_labels, occ_preds, average="macro", labels=list(range(1,13)), zero_division=0)
rec_occ      = recall_score(occ_labels, occ_preds,   average="macro", labels=list(range(1,13)), zero_division=0)

# ── Board-level accuracy: % of boards where ALL 64 cells correct ─────────────
all_preds_boards  = cnn_preds.reshape(-1, 64) if len(cnn_preds) % 64 == 0 else None

print("═" * 60)
print("CNN EVALUATION RESULTS")
print("═" * 60)
print(f"\n── Full 13-class (empty + 12 pieces) ──")
print(f"  Accuracy         : {acc_full:.4f} ({acc_full*100:.2f}%)")
print(f"  Precision (macro): {prec_macro:.4f}")
print(f"  Recall    (macro): {rec_macro:.4f}")
print(f"  F1 (macro)       : {f1_macro:.4f}")
print(f"  F1 (weighted)    : {f1_weighted:.4f}")
print(f"\n── Occupied cells only (12 piece types) ──")
print(f"  Accuracy         : {acc_occ:.4f} ({acc_occ*100:.2f}%)")
print(f"  Precision (macro): {prec_occ:.4f}")
print(f"  Recall    (macro): {rec_occ:.4f}")
print(f"  F1 (macro)       : {f1_occ_macro:.4f}")
print(f"  F1 (weighted)    : {f1_occ_w:.4f}")

In [ ]:
print("\n── Per-class report (all 13 classes) ──")
print(classification_report(
    cnn_labels, cnn_preds,
    labels=list(range(13)),
    target_names=CNN_CLASSES,
    digits=4,
    zero_division=0,
))

### 7.4 — Confusion matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(cnn_labels, cnn_preds, labels=list(range(13)))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(1)

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, data, title, fmt in zip(
    axes,
    [cm, cm_norm],
    ["Confusion Matrix (counts)", "Confusion Matrix (row-normalized)"],
    ["d", ".2f"],
):
    sns.heatmap(
        data, ax=ax,
        xticklabels=CNN_CLASSES, yticklabels=CNN_CLASSES,
        cmap="Blues", annot=(data.shape[0] <= 13), fmt=fmt,
        linewidths=0.3, linecolor="#ddd",
        cbar_kws={"shrink": 0.8},
    )
    ax.set_title(title, fontsize=13, pad=10)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=45)

plt.suptitle("CNN  — 13-class confusion matrix (test set)", fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig("/content/cnn_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → /content/cnn_confusion_matrix.png")

### 7.5 — Per-class F1 bar chart

In [ ]:
from sklearn.metrics import f1_score as _f1

per_class_f1 = _f1(
    cnn_labels, cnn_preds,
    average=None, labels=list(range(13)), zero_division=0
)
per_class_prec = precision_score(
    cnn_labels, cnn_preds,
    average=None, labels=list(range(13)), zero_division=0
)
per_class_rec = recall_score(
    cnn_labels, cnn_preds,
    average=None, labels=list(range(13)), zero_division=0
)

x = np.arange(13)
w = 0.26
fig, ax = plt.subplots(figsize=(15, 5))
ax.bar(x - w, per_class_prec, w, label="Precision", color="#4C72B0")
ax.bar(x,     per_class_rec,  w, label="Recall",    color="#55A868")
ax.bar(x + w, per_class_f1,   w, label="F1",        color="#C44E52")

ax.set_xticks(x)
ax.set_xticklabels(CNN_CLASSES, rotation=40, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("CNN — Per-class Precision / Recall / F1  (test set)", fontsize=13)
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig("/content/cnn_per_class_f1.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → /content/cnn_per_class_f1.png")

### 7.6 — Latency

In [ ]:
@torch.no_grad()
def measure_latency_ms(model, input_shape, device, warmup=20, iters=100):
    model = model.to(device).eval()
    x = torch.rand(*input_shape, device=device)
    for _ in range(warmup):
        model(x)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters):
        model(x)
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / iters * 1000.0


# Single-cell latency (batch=1)
lat_cell_single = measure_latency_ms(cnn, (1, 3, 50, 50), DEVICE)
# Batch latency (realistic: batch of 64 cells = 1 board)
lat_cell_batch64 = measure_latency_ms(cnn, (64, 3, 50, 50), DEVICE)

print(f"CNN latency (1 cell, batch=1)    : {lat_cell_single:.3f} ms")
print(f"CNN latency (64 cells, 1 board)  : {lat_cell_batch64:.3f} ms")
print(f"Effective throughput (boards/s)  : {1000/lat_cell_batch64:.1f}")

CNN_LATENCY_MS = lat_cell_batch64   # used for summary

## 8 · YOLO evaluation

### 8.1 — YOLOv8n  (primary — pretrained backbone)
Runs `model.val()` on the full test split with ground-truth cell-box labels.

In [ ]:
from ultralytics import YOLO

yolo_n = YOLO(str(YOLO_N_WEIGHTS))

print(f"YOLOv8n parameters: {sum(p.numel() for p in yolo_n.model.parameters()):,}")

metrics_n = yolo_n.val(
    data=str(DATA_YAML),
    split="val",         # 'val' key in data.yaml points to images/test
    imgsz=400,
    batch=32,
    device=DEVICE,
    verbose=True,
    save_json=False,
    plots=True,
)

In [ ]:
def print_yolo_metrics(metrics, name):
    box = metrics.box
    print(f"\n{'═'*55}")
    print(f" YOLO — {name}")
    print(f"{'═'*55}")
    print(f"  mAP@50       : {box.map50:.4f}")
    print(f"  mAP@50-95    : {box.map:.4f}")
    print(f"  Precision (mean) : {box.mp:.4f}")
    print(f"  Recall    (mean) : {box.mr:.4f}")

    # Speed
    spd = metrics.speed   # dict: preprocess / inference / postprocess (ms/img)
    total_ms = spd.get("preprocess", 0) + spd.get("inference", 0) + spd.get("postprocess", 0)
    print(f"  Latency (ms/board): {total_ms:.2f}  "
          f"(pre={spd.get('preprocess',0):.2f} "
          f"inf={spd.get('inference',0):.2f} "
          f"post={spd.get('postprocess',0):.2f})")
    print(f"  Throughput (boards/s): {1000/max(total_ms,0.001):.1f}")

    # Per-class
    try:
        cls_idx = metrics.ap_class_index
        aps50   = box.ap50
        aps     = box.ap
        precs   = box.p
        recs    = box.r
        print(f"\n  {'Class':<8} {'P':>7} {'R':>7} {'AP@50':>8} {'AP@50-95':>10}")
        print(f"  {'-'*45}")
        for i, ci in enumerate(cls_idx):
            cls_name = YOLO_CLASSES[ci] if ci < len(YOLO_CLASSES) else str(ci)
            print(f"  {cls_name:<8} {precs[i]:>7.4f} {recs[i]:>7.4f} "
                  f"{aps50[i]:>8.4f} {aps[i]:>10.4f}")
    except Exception as e:
        print(f"  (per-class breakdown unavailable: {e})")


print_yolo_metrics(metrics_n, "yolov8n")
YOLO_N_LATENCY = (
    metrics_n.speed.get("preprocess", 0) +
    metrics_n.speed.get("inference",  0) +
    metrics_n.speed.get("postprocess",0)
)

### 8.2 — YOLOv8s  (larger variant)

In [ ]:
yolo_s = YOLO(str(YOLO_S_WEIGHTS))
print(f"YOLOv8s parameters: {sum(p.numel() for p in yolo_s.model.parameters()):,}")

metrics_s = yolo_s.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=400,
    batch=16,
    device=DEVICE,
    verbose=False,
)
print_yolo_metrics(metrics_s, "yolov8s")
YOLO_S_LATENCY = (
    metrics_s.speed.get("preprocess", 0) +
    metrics_s.speed.get("inference",  0) +
    metrics_s.speed.get("postprocess",0)
)

### 8.3 — YOLO-pico  (from-scratch, lightweight baseline)

In [ ]:
yolo_pico = YOLO(str(YOLO_PICO_WEIGHTS))
print(f"YOLO-pico parameters: {sum(p.numel() for p in yolo_pico.model.parameters()):,}")

metrics_pico = yolo_pico.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=400,
    batch=32,
    device=DEVICE,
    verbose=False,
)
print_yolo_metrics(metrics_pico, "yolo_pico")
YOLO_PICO_LATENCY = (
    metrics_pico.speed.get("preprocess", 0) +
    metrics_pico.speed.get("inference",  0) +
    metrics_pico.speed.get("postprocess",0)
)

### 8.4 — YOLO per-class F1 chart (yolov8n)

In [ ]:
try:
    cls_idx = metrics_n.ap_class_index
    yolo_precs = metrics_n.box.p
    yolo_recs  = metrics_n.box.r
    yolo_f1    = 2 * yolo_precs * yolo_recs / (yolo_precs + yolo_recs + 1e-9)
    yolo_names = [YOLO_CLASSES[ci] if ci < len(YOLO_CLASSES) else str(ci)
                  for ci in cls_idx]

    x = np.arange(len(yolo_names))
    w = 0.26
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(x - w, yolo_precs, w, label="Precision", color="#4C72B0")
    ax.bar(x,     yolo_recs,  w, label="Recall",    color="#55A868")
    ax.bar(x + w, yolo_f1,    w, label="F1",        color="#C44E52")
    ax.set_xticks(x)
    ax.set_xticklabels(yolo_names, rotation=40, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("YOLOv8n — Per-class Precision / Recall / F1  (test set)", fontsize=13)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("/content/yolo_per_class_f1.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved → /content/yolo_per_class_f1.png")
except Exception as e:
    print(f"Could not plot per-class YOLO metrics: {e}")

## 9 · Model efficiency profiling

In [ ]:
# FLOP count via thop
def count_flops(model, input_shape):
    try:
        import copy
        from thop import profile
        dev  = next(model.parameters()).device
        prob = copy.deepcopy(model)
        x    = torch.rand(*input_shape, device=dev)
        flops, _ = profile(prob, inputs=(x,), verbose=False)
        return int(flops)
    except Exception as e:
        return f"N/A ({e})"


cnn_params = sum(p.numel() for p in cnn.parameters())
cnn_flops  = count_flops(cnn, (1, 3, 50, 50))

yolo_n_params = sum(p.numel() for p in yolo_n.model.parameters())
yolo_s_params = sum(p.numel() for p in yolo_s.model.parameters())
pico_params   = sum(p.numel() for p in yolo_pico.model.parameters())

print(f"{'Model':<18} {'Params':>12} {'FLOPs (single input)':>22}")
print("-" * 55)
print(f"{'Custom CNN':<18} {cnn_params:>12,} {str(cnn_flops):>22}")
print(f"{'YOLO-pico':<18} {pico_params:>12,} {'(400×400 board)':>22}")
print(f"{'YOLOv8n':<18} {yolo_n_params:>12,} {'(400×400 board)':>22}")
print(f"{'YOLOv8s':<18} {yolo_s_params:>12,} {'(400×400 board)':>22}")

## 10 · Comparison summary

In [ ]:
# Build comparison rows
def _fmt(v, pct=False):
    if isinstance(v, float):
        return f"{v*100:.2f}%" if pct else f"{v:.4f}"
    return str(v)


rows = [
    {
        "Model"         : "Custom CNN",
        "Task"          : "Cell classify (13-cls)",
        "Accuracy"      : acc_full,
        "Occ. Accuracy" : acc_occ,
        "F1 macro"      : f1_macro,
        "F1 weighted"   : f1_weighted,
        "Precision"     : prec_macro,
        "Recall"        : rec_macro,
        "mAP@50"        : "—",
        "mAP@50-95"     : "—",
        "Params"        : f"{cnn_params:,}",
        "Latency (ms)"  : f"{CNN_LATENCY_MS:.2f}",
    },
    {
        "Model"         : "YOLO-pico",
        "Task"          : "Detection (12-cls)",
        "Accuracy"      : "—",
        "Occ. Accuracy" : "—",
        "F1 macro"      : "—",
        "F1 weighted"   : "—",
        "Precision"     : metrics_pico.box.mp,
        "Recall"        : metrics_pico.box.mr,
        "mAP@50"        : metrics_pico.box.map50,
        "mAP@50-95"     : metrics_pico.box.map,
        "Params"        : f"{pico_params:,}",
        "Latency (ms)"  : f"{YOLO_PICO_LATENCY:.2f}",
    },
    {
        "Model"         : "YOLOv8n",
        "Task"          : "Detection (12-cls)",
        "Accuracy"      : "—",
        "Occ. Accuracy" : "—",
        "F1 macro"      : "—",
        "F1 weighted"   : "—",
        "Precision"     : metrics_n.box.mp,
        "Recall"        : metrics_n.box.mr,
        "mAP@50"        : metrics_n.box.map50,
        "mAP@50-95"     : metrics_n.box.map,
        "Params"        : f"{yolo_n_params:,}",
        "Latency (ms)"  : f"{YOLO_N_LATENCY:.2f}",
    },
    {
        "Model"         : "YOLOv8s",
        "Task"          : "Detection (12-cls)",
        "Accuracy"      : "—",
        "Occ. Accuracy" : "—",
        "F1 macro"      : "—",
        "F1 weighted"   : "—",
        "Precision"     : metrics_s.box.mp,
        "Recall"        : metrics_s.box.mr,
        "mAP@50"        : metrics_s.box.map50,
        "mAP@50-95"     : metrics_s.box.map,
        "Params"        : f"{yolo_s_params:,}",
        "Latency (ms)"  : f"{YOLO_S_LATENCY:.2f}",
    },
]

# Print table
cols = list(rows[0].keys())
widths = {c: max(len(c), max(len(str(r[c])) for r in rows)) for c in cols}
sep = "  ".join("-" * widths[c] for c in cols)

print("\n" + "="*120)
print("FULL COMPARISON SUMMARY")
print("="*120)
header = "  ".join(f"{c:<{widths[c]}}" for c in cols)
print(header)
print(sep)
for r in rows:
    line = "  ".join(
        f"{(_fmt(v, pct=False) if isinstance(v,float) else str(v)):<{widths[c]}}"
        for c, v in r.items()
    )
    print(line)
print("="*120)

### 10.1 — Comparison bar charts

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Accuracy / mAP comparison ─────────────────────────────────────────────────
ax = axes[0]
models_acc = ["CNN\n(all 13 cls)", "CNN\n(occ only)"]
vals_acc   = [acc_full, acc_occ]
bars = ax.bar(models_acc, vals_acc, color=["#4C72B0", "#4C72B0"])
ax.bar_label(bars, fmt="%.4f", padding=3)
ax.set_ylim(0, 1.1)
ax.set_title("CNN Accuracy", fontsize=12)
ax.set_ylabel("Score")
ax.grid(axis="y", alpha=0.3)

# ── mAP@50 across YOLO variants ──────────────────────────────────────────────
ax = axes[1]
yolo_names = ["YOLO-pico", "YOLOv8n", "YOLOv8s"]
map50_vals  = [metrics_pico.box.map50, metrics_n.box.map50, metrics_s.box.map50]
bars = ax.bar(yolo_names, map50_vals, color=["#DD8452", "#55A868", "#C44E52"])
ax.bar_label(bars, fmt="%.4f", padding=3)
ax.set_ylim(0, 1.1)
ax.set_title("YOLO mAP@50", fontsize=12)
ax.set_ylabel("mAP@50")
ax.grid(axis="y", alpha=0.3)

# ── Latency comparison ───────────────────────────────────────────────────────
ax = axes[2]
lat_names = ["CNN\n(64 cells)", "YOLO-pico", "YOLOv8n", "YOLOv8s"]
lat_vals  = [CNN_LATENCY_MS, YOLO_PICO_LATENCY, YOLO_N_LATENCY, YOLO_S_LATENCY]
bars = ax.bar(lat_names, lat_vals,
              color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"])
ax.bar_label(bars, fmt="%.2f ms", padding=3)
ax.set_title("Latency per board (ms)", fontsize=12)
ax.set_ylabel("ms")
ax.grid(axis="y", alpha=0.3)

plt.suptitle("Chess Piece Detection — Model Comparison (T4 GPU)", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("/content/model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → /content/model_comparison.png")

### 10.2 — Save full results to JSON

In [ ]:
import json

def _to_py(v):
    """Convert numpy scalars to plain Python for JSON serialization."""
    if isinstance(v, (np.floating, np.integer)):
        return v.item()
    return v


results = {
    "test_images": len(TEST_IMGS),
    "cnn": {
        "params"          : int(cnn_params),
        "latency_ms_board": float(CNN_LATENCY_MS),
        "all_classes": {
            "accuracy"        : float(acc_full),
            "f1_macro"        : float(f1_macro),
            "f1_weighted"     : float(f1_weighted),
            "precision_macro" : float(prec_macro),
            "recall_macro"    : float(rec_macro),
        },
        "occupied_cells_only": {
            "accuracy"        : float(acc_occ),
            "f1_macro"        : float(f1_occ_macro),
            "f1_weighted"     : float(f1_occ_w),
            "precision_macro" : float(prec_occ),
            "recall_macro"    : float(rec_occ),
        },
        "per_class_f1": {CNN_CLASSES[i]: float(per_class_f1[i]) for i in range(13)},
    },
    "yolo_pico": {
        "params"      : int(pico_params),
        "latency_ms"  : float(YOLO_PICO_LATENCY),
        "map50"       : float(metrics_pico.box.map50),
        "map50_95"    : float(metrics_pico.box.map),
        "precision"   : float(metrics_pico.box.mp),
        "recall"      : float(metrics_pico.box.mr),
    },
    "yolov8n": {
        "params"      : int(yolo_n_params),
        "latency_ms"  : float(YOLO_N_LATENCY),
        "map50"       : float(metrics_n.box.map50),
        "map50_95"    : float(metrics_n.box.map),
        "precision"   : float(metrics_n.box.mp),
        "recall"      : float(metrics_n.box.mr),
    },
    "yolov8s": {
        "params"      : int(yolo_s_params),
        "latency_ms"  : float(YOLO_S_LATENCY),
        "map50"       : float(metrics_s.box.map50),
        "map50_95"    : float(metrics_s.box.map),
        "precision"   : float(metrics_s.box.mp),
        "recall"      : float(metrics_s.box.mr),
    },
}

out_json = Path("/content/eval_results.json")
out_json.write_text(json.dumps(results, indent=2))
print(f"Results saved → {out_json}")
print(json.dumps(results, indent=2))

---
## Outputs summary

| File | Content |
|---|---|
| `/content/cnn_confusion_matrix.png` | 13-class confusion matrix (raw + row-normalized) |
| `/content/cnn_per_class_f1.png` | Per-class P / R / F1 bar chart |
| `/content/yolo_per_class_f1.png` | YOLOv8n per-class P / R / F1 |
| `/content/model_comparison.png` | Accuracy · mAP · latency side-by-side |
| `/content/eval_results.json` | All numeric results (machine-readable) |

Download via *Files* panel on the left sidebar.